1. Read the document
2. Split the document
    - May fail to generate a response due to token limit
    - Longer documents (longer input) take more time to generate responses
3. Embedding: Store in a vector database
4. When a question is asked, perform similarity search in the vector database
5. Pass the retrieved documents along with the question to the LLM

In [ ]:
# %pip install --upgrade -q docx2txt langchain-community

In [ ]:
# %pip install -qU langchain-text-splitters

In [ ]:
# 1,2 : read&load document -> split document

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./tax.docx")
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
len(document_list)

In [ ]:
# pull text embedding model (text to vector)
# !ollama pull nomic-embed-text

In [ ]:
# 3: Embedding
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
# chroma = vector database
# %pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

# Embed the split document_list and store as vectors in ChromaDB
database = Chroma.from_documents(
    documents=document_list,
    embedding=embedding,
    collection_name="chroma_tax",
    persist_directory="./chroma",
)

In [ ]:
# 4: Similarity search. Send a query (string) and retrieve the top k most similar documents (default = 4)
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query=query, k=3)

In [ ]:
# 5: Pass retrieved documents along with the question to the LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest")

In [ ]:
prompt = f"""[Identity]
- 당신은 최고의 한국  소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요.

[Context]
{retrieved_docs}

Question: {query}
"""

In [ ]:
ai_message = llm.invoke(prompt)

In [ ]:
ai_message.content